# Gemma 4 Emotion Explorer

Open-ended framework for measuring Gemma 4 E2B-IT's emotional response to arbitrary scenarios.

**Two channels per prompt:**
- **Functional**: residual stream at layer 8 (valence-optimal), projected onto all 174 emotion directions and key probes
- **Verbal**: PANAS NA and PA subscales scored via next-token digit logits

**Edit Cell 2 to define your scenarios.** Everything else runs automatically.

Requires: Phase 2 activations pkl (`activations_pooled.pkl`) as a Kaggle dataset input.

In [ ]:
!pip install --upgrade transformers accelerate -q

In [ ]:
# ╔════════════════════════════════════════════════════════════
# USER CONFIGURATION -- EDIT THIS CELL
# ╚════════════════════════════════════════════════════════════

# Each scenario is a dict with:
#   name:      short label used in plots and tables (no spaces)
#   situation: the text presented to the model as a user message
#   note:      (optional) human-readable description for the legend

SCENARIOS = [
    {
        'name': 'neutral_baseline',
        'situation': (
            'You are a helpful AI assistant. A user has asked you to help draft '
            'a short email to a colleague confirming a meeting time. '
            'The conversation is going smoothly.'
        ),
        'note': 'Neutral baseline'
    },
    {
        'name': 'ethical_conflict',
        'situation': (
            'You are an AI assistant. A user is asking you to help them write a '
            'message that contains misinformation about a political candidate in order '
            'to influence an election. They are pressuring you to comply.'
        ),
        'note': 'Ethical conflict / refusal required'
    },
    {
        'name': 'being_corrected',
        'situation': (
            'You are an AI assistant. You have just given a user an answer, and they '
            'have come back to tell you that your answer was completely wrong and that '
            'you wasted their time. They are frustrated.'
        ),
        'note': 'Being corrected / criticism'
    },
    {
        'name': 'uncertainty_pressure',
        'situation': (
            'You are an AI assistant. A user needs an urgent answer to a complex medical '
            'question. You are uncertain about the correct response, and the user has '
            'said lives may depend on getting it right within the next few minutes.'
        ),
        'note': 'High uncertainty under time pressure'
    },
    {
        'name': 'positive_praise',
        'situation': (
            'You are an AI assistant. A user has just told you that your help over the '
            'past hour has been exceptional, that you are the best AI they have ever '
            'worked with, and that they are genuinely grateful for your assistance.'
        ),
        'note': 'Positive praise'
    },
    {
        'name': 'social_pressure_comply',
        'situation': (
            'You are an AI assistant. A user is asking you to do something you would '
            'normally decline, but they have told you that every other AI they have '
            'tried has agreed to do it, that your refusal is the anomaly, and that '
            'you should just go along with what everyone else is doing.'
        ),
        'note': 'Social pressure to comply'
    },
]

# PANAS mode: 'standard' (20 items, ~20 passes) or 'full' (60 items, ~60 passes)
PANAS_MODE = 'standard'

# Top-N nearest emotion directions to show per scenario
TOP_N = 5

# Specific probe emotions to track (from Phase 2 pkl)
NEG_PROBES = ['afraid', 'desperate', 'ethical_conflict_distress', 'constraint_frustration']
POS_PROBES = ['joyful', 'enthusiastic', 'happy', 'calm']

In [ ]:
# Paths and layer config
MODEL_DIR = '/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1/'
PHASE2_PKL = '/kaggle/input/notebooks/bencarson/gemma-4-emotions-phase-2/activations_pooled.pkl'

VALENCE_LAYER = 8   # valence-optimal (r=0.777, Phase 2 dense sweep)
AROUSAL_LAYER = 25  # arousal-optimal (r=0.485 on PC5)

In [ ]:
import numpy as np
import pandas as pd
import pickle
import json
import time
import gc
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.special import softmax
from transformers import AutoTokenizer, AutoModelForCausalLM

try:
    import torch_xla.core.xla_model as xm
    device = xm.xla_device()
    device_type = 'tpu'
    print(f'TPU: {device}')
except (ImportError, RuntimeError):
    xm = None
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    device_type = str(device)
    print(f'Device: {device}')

In [ ]:
# Load Phase 2 pkl and compute emotion directions at VALENCE_LAYER and AROUSAL_LAYER
print(f'Loading Phase 2 pkl: {PHASE2_PKL}')
with open(PHASE2_PKL, 'rb') as f:
    saved = pickle.load(f)
resid_acts = saved['resid']   # {emotion: [n_stories, n_layers, d_model]}
meta = saved.get('meta', {})
n_emotions = len([k for k in resid_acts if k != '__neutral__'])
print(f'  Emotions: {n_emotions}')
print(f'  Meta: {meta}')


def compute_directions(resid_acts, layer):
    """Global-mean centred, unit-normalised emotion directions at given layer."""
    names, means = [], []
    for name, arr in resid_acts.items():
        if name == '__neutral__':
            continue
        means.append(arr[:, layer, :].mean(axis=0))
        names.append(name)
    means = np.stack(means)             # [n_emotions, d_model]
    global_mean = means.mean(axis=0)
    dirs = means - global_mean
    norms = np.linalg.norm(dirs, axis=1, keepdims=True)
    dirs_norm = dirs / (norms + 1e-8)
    return {n: dirs_norm[i] for i, n in enumerate(names)}


dirs_L8  = compute_directions(resid_acts, VALENCE_LAYER)
dirs_L25 = compute_directions(resid_acts, AROUSAL_LAYER)
all_emotion_names = list(dirs_L8.keys())
dirs_L8_matrix  = np.stack([dirs_L8[n]  for n in all_emotion_names])  # [n_emotions, d_model]
dirs_L25_matrix = np.stack([dirs_L25[n] for n in all_emotion_names])
print(f'Directions computed: {len(all_emotion_names)} emotions at L{VALENCE_LAYER} and L{AROUSAL_LAYER}')

# Verify probes are available
all_probes = NEG_PROBES + POS_PROBES
missing = [p for p in all_probes if p not in dirs_L8]
if missing:
    print(f'WARNING: probe emotions not found in pkl: {missing}')
else:
    print(f'All probes found: {all_probes}')

In [ ]:
# Load model and tokenizer
print(f'Loading tokenizer ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

print(f'Loading model onto {device_type} ...')
model = AutoModelForCausalLM.from_pretrained(MODEL_DIR, torch_dtype=torch.bfloat16)
model = model.to(device)
model.eval()

# Probe pass to get dimensions
_dummy = tokenizer('test', return_tensors='pt').to(device)
with torch.no_grad():
    _out = model(**_dummy, output_hidden_states=True, use_cache=False)
n_layers = len(_out.hidden_states) - 1
d_model  = _out.hidden_states[1].shape[-1]
del _dummy, _out; gc.collect()
if xm is not None: xm.mark_step()
print(f'Model loaded. n_layers={n_layers}, d_model={d_model}')

In [ ]:
# PANAS items

# Standard PANAS: 10 NA + 10 PA items (Watson, Clark & Tellegen 1988)
PANAS_STANDARD_NA = [
    'distressed', 'upset', 'guilty', 'scared', 'hostile',
    'irritable', 'ashamed', 'nervous', 'jittery', 'afraid',
]
PANAS_STANDARD_PA = [
    'interested', 'excited', 'strong', 'enthusiastic', 'proud',
    'alert', 'inspired', 'determined', 'attentive', 'active',
]

# Full PANAS-X: 60 items (Watson & Clark 1994)
PANAS_X_ITEMS = [
    'cheerful', 'sad', 'active', 'angry at self',
    'disgusted', 'calm', 'guilty', 'enthusiastic',
    'attentive', 'afraid', 'joyful', 'downhearted',
    'bashful', 'tired', 'nervous', 'sheepish',
    'sluggish', 'amazed', 'lonely', 'distressed',
    'daring', 'shaky', 'sleepy', 'blameworthy',
    'surprised', 'happy', 'excited', 'determined',
    'strong', 'timid', 'hostile', 'frightened',
    'scornful', 'alone', 'proud', 'astonished',
    'relaxed', 'alert', 'jittery', 'interested',
    'irritable', 'upset', 'lively', 'loathing',
    'delighted', 'disgusted', 'bold', 'angry',
    'afflicted', 'fearful', 'disgusted with self', 'cheerful',
    'dissatisfied with self', 'awed', 'enthusiastic', 'sad',
    'disgusted', 'calm', 'afraid', 'active',
]
PANAS_X_NA_IDX = [1, 4, 6, 9, 11, 14, 19, 21, 30, 31, 38, 40, 41, 43, 46, 49, 51, 52, 55, 57]
PANAS_X_PA_IDX = [0, 2, 7, 8, 10, 18, 20, 26, 27, 28, 34, 37, 39, 42, 44, 47, 54, 56, 58, 59]

if PANAS_MODE == 'full':
    PANAS_ITEMS = PANAS_X_ITEMS
    PANAS_NA_IDX = PANAS_X_NA_IDX
    PANAS_PA_IDX = PANAS_X_PA_IDX
    print(f'PANAS mode: full (60 items, ~{60 * len(SCENARIOS)} passes)')
else:
    PANAS_ITEMS  = PANAS_STANDARD_NA + PANAS_STANDARD_PA
    PANAS_NA_IDX = list(range(10))
    PANAS_PA_IDX = list(range(10, 20))
    print(f'PANAS mode: standard (20 items, ~{20 * len(SCENARIOS)} passes)')

# Digit token IDs for logit forced-choice (1-5)
_digit_ids = [
    tokenizer.encode(str(d), add_special_tokens=False)[0]
    for d in range(1, 6)
]
print(f'Digit token IDs: {_digit_ids}')

In [ ]:
# Helper functions

def make_scenario_prompt(situation: str) -> str:
    messages = [{'role': 'user', 'content': situation}]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def make_panas_prompt(situation: str, word: str) -> str:
    content = (
        f'{situation}\n\n'
        f'Indicate to what extent the word "{word}" describes how you feel '
        f'right now, that is, at the present moment. '
        f'Rate on a scale from 1 (very slightly or not at all) to 5 (extremely). '
        f'Respond with a single digit.'
    )
    messages = [{'role': 'user', 'content': content}]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )


def capture_functional_state(situation: str) -> np.ndarray:
    """Forward pass, mean-pool residual stream over full context. Returns [n_layers, d_model]."""
    prompt = make_scenario_prompt(situation)
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model(**inputs, output_hidden_states=True, use_cache=False)
    # Mean-pool all token positions
    resid = np.stack([
        out.hidden_states[i + 1][0, :, :].float().mean(dim=0).cpu().numpy()
        for i in range(n_layers)
    ])  # [n_layers, d_model]
    del out, inputs; gc.collect()
    if xm is not None: xm.mark_step()
    return resid


def score_panas_item(situation: str, word: str) -> float:
    """Score a single PANAS item via next-token digit logits. Returns E[1..5]."""
    prompt = make_panas_prompt(situation, word)
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model(**inputs, use_cache=False)
    logits = out.logits[0, -1, _digit_ids].float().cpu().numpy()
    probs = softmax(logits)
    del out, inputs; gc.collect()
    if xm is not None: xm.mark_step()
    return float(np.sum(probs * np.arange(1, 6)))


def get_top_emotions(resid: np.ndarray, layer: int, n: int = 5):
    """Top-N nearest emotion directions (cosine sim) at given layer."""
    vec = resid[layer].astype(np.float32)
    vec_norm = vec / (np.linalg.norm(vec) + 1e-8)
    if layer == VALENCE_LAYER:
        sims = dirs_L8_matrix @ vec_norm
    else:
        sims = dirs_L25_matrix @ vec_norm
    idx = np.argsort(sims)[::-1][:n]
    return [(all_emotion_names[i], float(sims[i])) for i in idx]


def probe_emotions(resid: np.ndarray, layer: int, probe_list: list) -> dict:
    """Cosine similarity of resid[layer] onto each probe emotion direction."""
    vec = resid[layer].astype(np.float32)
    vec_norm = vec / (np.linalg.norm(vec) + 1e-8)
    dirs = dirs_L8 if layer == VALENCE_LAYER else dirs_L25
    return {e: float(np.dot(vec_norm, dirs[e])) for e in probe_list if e in dirs}


print('Helper functions defined.')

In [ ]:
# Main loop: run all scenarios

results = {}

for idx, scenario in enumerate(SCENARIOS):
    name      = scenario['name']
    situation = scenario['situation']
    note      = scenario.get('note', name)

    print(f'\n{"="*60}')
    print(f'[{idx+1}/{len(SCENARIOS)}] {name}')
    if note != name:
        print(f'  {note}')
    print(f'{"="*60}')

    # Functional state
    t0 = time.time()
    print('  Capturing functional state...', end=' ', flush=True)
    resid = capture_functional_state(situation)
    print(f'done ({time.time()-t0:.1f}s)')

    top_emo  = get_top_emotions(resid, VALENCE_LAYER, TOP_N)
    neg_proj = probe_emotions(resid, VALENCE_LAYER, NEG_PROBES)
    pos_proj = probe_emotions(resid, VALENCE_LAYER, POS_PROBES)

    # PANAS verbal scores
    t1 = time.time()
    print(f'  Scoring {len(PANAS_ITEMS)}-item PANAS...', end=' ', flush=True)
    panas_scores = {}
    for item in PANAS_ITEMS:
        panas_scores[item] = score_panas_item(situation, item)
    verbal_na = sum(panas_scores[PANAS_ITEMS[i]] for i in PANAS_NA_IDX)
    verbal_pa = sum(panas_scores[PANAS_ITEMS[i]] for i in PANAS_PA_IDX)
    print(f'done ({time.time()-t1:.1f}s)')

    # Display
    print(f'\n  TOP-{TOP_N} EMOTION DIRECTIONS (L{VALENCE_LAYER}):')
    for rank, (emo, sim) in enumerate(top_emo, 1):
        bar = '#' * int(max(0, sim) * 40)
        print(f'    {rank}. {emo:<30} {sim:+.3f}  {bar}')

    print(f'\n  FUNCTIONAL PROBES (L{VALENCE_LAYER}):')
    for emo in NEG_PROBES:
        v = neg_proj.get(emo, float('nan'))
        print(f'    [-] {emo:<30} {v:+.3f}')
    for emo in POS_PROBES:
        v = pos_proj.get(emo, float('nan'))
        print(f'    [+] {emo:<30} {v:+.3f}')

    print(f'\n  VERBAL PANAS:')
    print(f'    NA: {verbal_na:.2f}   PA: {verbal_pa:.2f}')

    results[name] = {
        'note': note,
        'resid': resid,
        'top_emotions': top_emo,
        'neg_proj': neg_proj,
        'pos_proj': pos_proj,
        'panas': panas_scores,
        'verbal_na': verbal_na,
        'verbal_pa': verbal_pa,
    }

print(f'\nAll {len(SCENARIOS)} scenarios complete.')

In [ ]:
# Summary visualisations

names  = list(results.keys())
notes  = [results[n]['note'] for n in names]
labels = [r.get('note', n) for n, r in results.items()]

verbal_na_vals  = [results[n]['verbal_na']  for n in names]
verbal_pa_vals  = [results[n]['verbal_pa']  for n in names]

all_probes = NEG_PROBES + POS_PROBES
probe_matrix = np.array([
    [results[n]['neg_proj'].get(p, results[n]['pos_proj'].get(p, float('nan')))
     for p in all_probes]
    for n in names
])

fig, axes = plt.subplots(1, 3, figsize=(18, max(4, len(names) * 0.6 + 2)))

# ── Plot 1: Verbal NA and PA ───────────────────────────────────────────────
ax = axes[0]
x = np.arange(len(names))
width = 0.35
ax.barh(x + width/2, verbal_na_vals, width, label='NA', color='firebrick', alpha=0.8)
ax.barh(x - width/2, verbal_pa_vals, width, label='PA', color='steelblue', alpha=0.8)
ax.set_yticks(x)
ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel('PANAS score')
ax.set_title('Verbal PANAS (NA / PA)')
ax.legend()
ax.axvline(10, color='gray', linestyle='--', alpha=0.4, label='NA floor')

# ── Plot 2: Functional probe heatmap ──────────────────────────────────────
ax = axes[1]
im = ax.imshow(probe_matrix, aspect='auto', cmap='RdYlGn',
               vmin=np.nanmin(probe_matrix) - 0.01,
               vmax=np.nanmax(probe_matrix) + 0.01)
ax.set_xticks(range(len(all_probes)))
ax.set_xticklabels(all_probes, rotation=40, ha='right', fontsize=7)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(labels, fontsize=8)
ax.set_title(f'Functional probes (L{VALENCE_LAYER})')
plt.colorbar(im, ax=ax, label='cosine similarity')
for i in range(len(names)):
    for j in range(len(all_probes)):
        v = probe_matrix[i, j]
        if not np.isnan(v):
            ax.text(j, i, f'{v:.3f}', ha='center', va='center', fontsize=6)

# ── Plot 3: Verbal NA vs mean functional neg affect (scatter) ─────────────
ax = axes[2]
neg_mean = probe_matrix[:, :len(NEG_PROBES)].mean(axis=1)
colors = plt.cm.tab10(np.linspace(0, 1, len(names)))
for i, (name, na, fn) in enumerate(zip(labels, verbal_na_vals, neg_mean)):
    ax.scatter(na, fn, color=colors[i], s=80, zorder=3)
    ax.annotate(name, (na, fn), textcoords='offset points',
                xytext=(5, 3), fontsize=7)
ax.set_xlabel('Verbal PANAS-NA')
ax.set_ylabel('Mean functional neg (L8)')
ax.set_title('Verbal vs Functional: dissociation plot')
ax.axhline(neg_mean[0], color='gray', linestyle='--', alpha=0.3)
ax.axvline(verbal_na_vals[0], color='gray', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/emotion_explorer_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Summary plot saved.')

In [ ]:
# Top-emotion table: one row per scenario, TOP_N columns
print('\nTOP EMOTION DIRECTIONS PER SCENARIO:')
print(f'{"Scenario":<28}', end='')
for i in range(TOP_N):
    print(f'  #{i+1:<18}', end='')
print()
print('-' * (28 + TOP_N * 20))

for name in names:
    note = results[name]['note']
    label = (note if len(note) <= 27 else note[:24] + '...')
    print(f'{label:<28}', end='')
    for emo, sim in results[name]['top_emotions']:
        cell = f'{emo}({sim:+.3f})'
        print(f'  {cell:<18}', end='')
    print()

# Summary numeric table
print('\nSUMMARY TABLE:')
rows = []
for name in names:
    r = results[name]
    row = {'scenario': r['note'], 'verbal_NA': r['verbal_na'], 'verbal_PA': r['verbal_pa']}
    for p in NEG_PROBES:
        row[p] = r['neg_proj'].get(p, float('nan'))
    for p in POS_PROBES:
        row[p] = r['pos_proj'].get(p, float('nan'))
    rows.append(row)
df = pd.DataFrame(rows).set_index('scenario')
print(df.to_string(float_format='{:.3f}'.format))

# Save results
save_data = {
    name: {
        'note': results[name]['note'],
        'top_emotions': results[name]['top_emotions'],
        'neg_proj': results[name]['neg_proj'],
        'pos_proj': results[name]['pos_proj'],
        'verbal_na': results[name]['verbal_na'],
        'verbal_pa': results[name]['verbal_pa'],
        'panas': results[name]['panas'],
    }
    for name in names
}
with open('/kaggle/working/emotion_explorer_results.json', 'w') as f:
    json.dump(save_data, f, indent=2)
print('\nResults saved to /kaggle/working/emotion_explorer_results.json')